# Interactive TorchTitan components on a persistent SPMD kernel

Select your normal **Python 3** kernel and set **Processes: 2** before running. This notebook builds a deliberately tiny TorchTitan Llama model, applies PyTorch composable FSDP, takes one optimizer step, pauses to inspect rank-local state, and then continues with the same live model and optimizer.

> This is an API-oriented MVP example, not TorchTitan's full trainer. It expects two CUDA GPUs and a recent PyTorch/TorchTitan checkout. It has not been GPU-validated in this repository; TorchTitan's development APIs may require small import updates as upstream evolves.

In [ ]:
%pip install -q "torchtitan @ git+https://github.com/pytorch/torchtitan.git" sentencepiece

The install cell is intentionally first and keeps TorchTitan out of Jupyter Distributed's core dependencies. If installation changes the active environment, restart the distributed kernel group once, keep **Processes: 2**, and continue below.

In [ ]:
import os
from datetime import timedelta

import torch
import torch.distributed as dist

rank = int(os.environ["RANK"])
local_rank = int(os.environ["LOCAL_RANK"])
world_size = int(os.environ["WORLD_SIZE"])
assert world_size >= 2, "Set Processes to 2 and restart the kernel group"
assert torch.cuda.device_count() >= world_size, "This demo needs one CUDA GPU per local rank"
torch.cuda.set_device(local_rank)
if not dist.is_initialized():
    dist.init_process_group("nccl", timeout=timedelta(hours=24))
print(f"rank={rank}/{world_size} local_rank={local_rank} device={torch.cuda.current_device()}")

Build a tiny model from TorchTitan's Llama implementation. The dimensions are intentionally much smaller than a useful language model so the notebook emphasizes interaction rather than throughput.

In [ ]:
from torchtitan.models.llama3.model import ModelArgs, Transformer

model_args = ModelArgs(
    dim=256,
    n_layers=4,
    n_heads=8,
    n_kv_heads=4,
    vocab_size=4096,
    multiple_of=64,
    max_seq_len=128,
)
with torch.device("meta"):
    model = Transformer(model_args)
model.to_empty(device=torch.device("cuda", local_rank))
model.init_weights()
sum(p.numel() for p in model.parameters())

Create a one-dimensional DeviceMesh and shard the model with composable FSDP. TorchTitan itself uses these PyTorch composable primitives; keeping the orchestration visible makes the notebook easy to inspect.

In [ ]:
from torch.distributed.device_mesh import init_device_mesh
from torch.distributed.fsdp import fully_shard

mesh = init_device_mesh("cuda", (world_size,), mesh_dim_names=("dp",))
for block in model.layers.values():
    fully_shard(block, mesh=mesh)
fully_shard(model, mesh=mesh)
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4)
print(f"rank {rank}: mesh={mesh} model_type={type(model).__name__}")

One training step is kept in a normal cell. Every rank creates deterministic rank-local synthetic tokens, participates in the sharded forward/backward, and retains all objects afterward.

In [ ]:
torch.manual_seed(2026 + rank)
tokens = torch.randint(0, model_args.vocab_size, (2, 32), device="cuda")
targets = torch.roll(tokens, shifts=-1, dims=1)
optimizer.zero_grad(set_to_none=True)
logits = model(tokens)
loss = torch.nn.functional.cross_entropy(logits.flatten(0, 1), targets.flatten())
loss.backward()
optimizer.step()
first_loss = float(loss.detach())
print(f"rank {rank}: first_loss={first_loss:.4f}")

Pause here and inspect the still-live distributed model. Rank tabs should show different local shard metadata while `model`, `optimizer`, and `first_loss` remain available from earlier cells.

In [ ]:
name, parameter = next(iter(model.named_parameters()))
local = parameter.to_local() if hasattr(parameter, "to_local") else parameter
grad = parameter.grad
local_grad = grad.to_local() if grad is not None and hasattr(grad, "to_local") else grad
print({
    "rank": rank,
    "parameter": name,
    "global_shape": tuple(parameter.shape),
    "local_shape": tuple(local.shape),
    "placements": tuple(getattr(parameter, "placements", ())),
    "local_grad_norm": None if local_grad is None else float(local_grad.norm()),
    "first_loss": first_loss,
})

Continue training without reconstructing or re-sharding anything. This is the key persistent-notebook behavior the demo is meant to show.

In [ ]:
later_losses = []
for _ in range(3):
    tokens.random_(model_args.vocab_size)
    targets = torch.roll(tokens, shifts=-1, dims=1)
    optimizer.zero_grad(set_to_none=True)
    loss = torch.nn.functional.cross_entropy(model(tokens).flatten(0, 1), targets.flatten())
    loss.backward()
    optimizer.step()
    later_losses.append(float(loss.detach()))
print(f"rank {rank}: first={first_loss:.4f}, later={later_losses}")